## Cell 1 — Find the repo 

In [1]:
import os, glob

# Auto-find the repo root (works regardless of exact zip/folder name)
candidates = glob.glob('/kaggle/input/**/run.py', recursive=True)
if not candidates:
    raise FileNotFoundError('Could not find run.py. Make sure the repo dataset is added to this notebook.')

REPO = os.path.dirname(candidates[0])
print('Repo found at:', REPO)
print('Contents:', os.listdir(REPO))

Repo found at: /kaggle/input/datasets/varunvijay007/rhetorica-role-model/semantic-segmentation-master
Contents: ['infer.py', 'train.py', 'model', 'prepare_data.py', 'categories.txt', 'run.py', 'saved', 'LICENSE', 'README.md', 'infer', 'data']


## Cell 2 — Set up working directory in /kaggle/working

In [2]:
import shutil

WORK = '/kaggle/working/semseg'

# Copy repo to writable location (kaggle/input is read-only)
if os.path.exists(WORK):
    shutil.rmtree(WORK)
shutil.copytree(REPO, WORK)

os.chdir(WORK)
print('Working in:', os.getcwd())

# Delete placeholder __init__.py files (required by README)
for f in ['infer/data/__init__.py', 'saved/__init__.py']:
    if os.path.exists(f):
        os.remove(f)
        print('Deleted:', f)

# Ensure saved/ and infer/data/ dirs exist
os.makedirs('saved', exist_ok=True)
os.makedirs('infer/data', exist_ok=True)
print('Done.')

Working in: /kaggle/working/semseg
Deleted: infer/data/__init__.py
Deleted: saved/__init__.py
Done.


In [6]:
# Fix for numpy >= 1.24 incompatibility in batchify()
# Replaces np.array(x)[idx] with Python list indexing

train_py = '/kaggle/working/semseg/train.py'

with open(train_py, 'r') as f:
    code = f.read()

# Apply the fix
fixed = code.replace(
    'x = np.array(x)[idx]\n    y = np.array(y)[idx]',
    'x = [x[i] for i in idx]\n    y = [y[i] for i in idx]'
)

with open(train_py, 'w') as f:
    f.write(fixed)

# Confirm it worked
assert 'np.array(x)[idx]' not in fixed, 'Fix did not apply!'
print('train.py patched successfully')

train.py patched successfully


## Cell 3 — Check GPU and verify data

In [3]:
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU found. Go to Notebook Settings > Accelerator > GPU T4 x2 for faster training.')

# Verify the pretrained embeddings data is present
emb_files = os.listdir('data/pretrained_embeddings')
print(f'\nPretrained embedding files: {len(emb_files)} documents found')
print('Sample:', emb_files[:3])

Device: cuda
GPU: Tesla T4

Pretrained embedding files: 50 documents found
Sample: ['2008_P_8.txt', '2010_S_431.txt', '1996_T_169.txt']


## Cell 4 — Install dependencies

Kaggle has PyTorch, numpy, sklearn already.  
We only need `sent2vec` — but only for inference on NEW documents, not for training.  
We install it now so everything is ready.

In [4]:
# Install sent2vec (needed only at inference time for new documents)
# Kaggle is Linux so this compiles fine
!git clone https://github.com/epfml/sent2vec.git /tmp/sent2vec -q
!pip install /tmp/sent2vec -q

import sent2vec
print('sent2vec installed successfully')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
sent2vec installed successfully


## Cell 5 — Train the best model

- Uses `data/pretrained_embeddings/` — 200-dim sent2vec embeddings already in the repo  
- Runs 5-fold cross-validation × 300 epochs  
- Saves best checkpoint per fold in `saved/`  
- Watch the `Val_F1` column — note which fold number is highest

In [7]:
!python run.py \
    --pretrained True \
    --data_path data/pretrained_embeddings/ \
    --save_path saved/ \
    --device {DEVICE} \
    --epochs 300 \
    --batch_size 32 \
    --lr 0.01 \
    --print_every 10


Preparing data ... Done
Vocabulary size: 2
#Tags: 10

Cross-validation


Initializing model ... Done

Evaluating on fold 0 ...
  EPOCH     Tr_LOSS   Tr_F1    Val_LOSS  Val_F1
-----------------------------------------------------------
     10    2626.018   0.572    1293.591   0.561
     20    1487.947   0.781    1019.448   0.673
     30     894.070   0.889     983.599   0.741
     40    1290.446   0.822     882.437   0.698
     50     658.229   0.916     915.559   0.730
     60     445.254   0.953     977.947   0.704
     70     391.090   0.953    1010.894   0.717
     80     266.671   0.971     993.256   0.723
     90     178.256   0.984    1227.273   0.757
    100     197.425   0.976    1149.597   0.735
    110     121.379   0.989    1278.354   0.701
    120      83.387   0.993    1361.281   0.742
    130      59.438   0.996    1470.006   0.714
    140      50.068   0.997    1585.628   0.729
    150      35.168   0.999    1551.149   0.743
    160      31.047   0.999    1589.957   0.

## Cell 6 — Find the best fold automatically

In [8]:
import json
from sklearn.metrics import f1_score

best_fold = -1
best_f1 = -1

print(f"{'Fold':<6} {'Val Macro-F1':>12}")
print('-' * 20)

for fold in range(5):
    path = f'saved/data_state{fold}.json'
    if not os.path.exists(path):
        print(f'  {fold}      not found')
        continue
    with open(path) as f:
        state = json.load(f)

    gold = [g for doc in state['gold'] for g in doc]
    pred = [p for doc in state['pred'] for p in doc]
    f1 = f1_score(gold, pred, average='macro')

    print(f'  {fold}        {f1:.4f}')

    if f1 > best_f1:
        best_f1 = f1
        best_fold = fold

print(f'\nBest fold: {best_fold}  |  Macro-F1: {best_f1:.4f}')

Fold   Val Macro-F1
--------------------
  0        0.7575
  1        0.7868
  2        0.7806
  3        0.8352
  4        0.8297

Best fold: 3  |  Macro-F1: 0.8352


## Cell 7 — Copy best model into infer/

In [9]:
shutil.copy(f'saved/model_state{best_fold}.tar', 'infer/model.tar')
shutil.copy('saved/word2idx.json', 'infer/word2idx.json')
shutil.copy('saved/tag2idx.json', 'infer/tag2idx.json')

print(f'Copied fold {best_fold} model to infer/')
print('infer/ contents:', os.listdir('infer'))

Copied fold 3 model to infer/
infer/ contents: ['model.tar', 'tag2idx.json', 'word2idx.json', 'data']


## Cell 8 — Download the pretrained sent2vec binary

This is the 2 GB model used to embed NEW documents at inference time.  
Skip this cell if you only want to test with the provided dataset.

In [10]:
!wget -q --show-progress \
    http://cse.iitkgp.ac.in/~saptarshi/models/sent2vec.bin \
    -O infer/sent2vec.bin

size_gb = os.path.getsize('infer/sent2vec.bin') / (1024**3)
print(f'Downloaded: {size_gb:.2f} GB')

infer/sent2vec.bin  100%[===================>]   2.10G  9.85MB/s    in 4m 4s   
Downloaded: 2.10 GB


In [13]:
# Fix for PyTorch 2.6 — torch.load now defaults to weights_only=True
# which blocks loading checkpoints that contain model architecture objects

infer_py = '/kaggle/working/semseg/infer.py'

with open(infer_py, 'r') as f:
    code = f.read()

fixed = code.replace(
    'ckpt = torch.load(args.model_path)',
    'ckpt = torch.load(args.model_path, weights_only=False)'
)

with open(infer_py, 'w') as f:
    f.write(fixed)

assert 'weights_only=False' in fixed, 'Fix did not apply!'
print('infer.py patched successfully')

infer.py patched successfully


## Cell 9 — Add your documents and run inference

Put your `.txt` files in `infer/data/` — one sentence per line, no labels needed.

In [11]:
# Write a sample test document
sample = """The appellant was convicted by the Sessions Court in 2001 for murder.
The High Court upheld the conviction on appeal in 2003.
The appellant contends that the evidence was circumstantial and insufficient.
Section 302 IPC prescribes punishment for murder.
In State vs. Mohan 1994, this Court held that circumstantial evidence must be conclusive.
We are of the opinion that the chain of circumstances is not complete.
The appeal is therefore allowed and the conviction is set aside.
"""

with open('infer/data/test_case.txt', 'w') as f:
    f.write(sample)

print('Test document written to infer/data/test_case.txt')
print('(You can also copy your own .txt files into infer/data/)')

Test document written to infer/data/test_case.txt
(You can also copy your own .txt files into infer/data/)


In [14]:
# Run inference
# --pretrained True  → uses sent2vec to embed each sentence at runtime
# Requires infer/sent2vec.bin (downloaded in Cell 8)

!python infer.py \
    --pretrained True \
    --device {DEVICE} \
    --data_path infer/data/ \
    --model_path infer/model.tar \
    --sent2vec_model_path infer/sent2vec.bin \
    --word2idx_path infer/word2idx.json \
    --tag2idx_path infer/tag2idx.json \
    --save_path infer/predictions.txt

Loading pretrained sent2vec model ... Done

Preparing data ... Done

Loading model ... Done
Saving predictions ... Done


## Cell 10 — Read and display predictions

In [15]:
with open('infer/predictions.txt') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        doc_name, labels = line.split('\t')
        label_list = labels.split(',')

        print(f'=== {doc_name} ===')

        doc_path = f'infer/data/{doc_name}.txt'
        if os.path.exists(doc_path):
            with open(doc_path) as df:
                sents = [s.strip() for s in df if s.strip()]
            for sent, label in zip(sents, label_list):
                print(f'  [{label:<30}]  {sent}')
        else:
            for i, lbl in enumerate(label_list):
                print(f'  Sentence {i+1}: {lbl}')
        print()

=== test_case ===
  [Facts                         ]  The appellant was convicted by the Sessions Court in 2001 for murder.
  [Facts                         ]  The High Court upheld the conviction on appeal in 2003.
  [Facts                         ]  The appellant contends that the evidence was circumstantial and insufficient.
  [Argument                      ]  Section 302 IPC prescribes punishment for murder.
  [Argument                      ]  In State vs. Mohan 1994, this Court held that circumstantial evidence must be conclusive.
  [Ratio of the decision         ]  We are of the opinion that the chain of circumstances is not complete.
  [Ratio of the decision         ]  The appeal is therefore allowed and the conviction is set aside.



## Cell 11 — Load model for use in your own code

In [22]:
import sys, json, torch
sys.path.insert(0, WORK)

from model.Hier_BiLSTM_CRF import Hier_LSTM_CRF_Classifier

with open('infer/word2idx.json') as f:
    word2idx = json.load(f)
with open('infer/tag2idx.json') as f:
    tag2idx = json.load(f)

idx2tag = {v: k for k, v in tag2idx.items()}
print('Labels:', [k for k in tag2idx if k not in ['<pad>','<start>','<end>']])

# Fix for PyTorch 2.6 — add weights_only=False
ckpt = torch.load('infer/model.tar', map_location=DEVICE, weights_only=False)

model = Hier_LSTM_CRF_Classifier(
    n_tags       = len(tag2idx),
    sent_emb_dim = 200,
    sos_tag_idx  = tag2idx['<start>'],
    eos_tag_idx  = tag2idx['<end>'],
    pad_tag_idx  = tag2idx['<pad>'],
    vocab_size   = len(word2idx),
    pretrained   = True,
    device       = DEVICE
).to(DEVICE)

model.load_state_dict(ckpt['state_dict'])
model.eval()
print('Model loaded successfully!')

Labels: ['Facts', 'Argument', 'Ratio of the decision', 'Statute', 'Precedent', 'Ruling by Present Court', 'Ruling by Lower Court']
Model loaded successfully!


In [23]:
# predict() helper — call this anywhere with a list of sentence strings
import sent2vec as sv

s2v = sv.Sent2vecModel()
s2v.load_model('infer/sent2vec.bin')

def predict(sentences):
    """sentences: list[str] → list[str] of predicted rhetorical role labels"""
    doc_x = [s2v.embed_sentence(s).flatten().tolist()[:200] for s in sentences]
    with torch.no_grad():
        preds = model([doc_x])[0]
    return [idx2tag[p] for p in preds]


# Quick test
sentences = [
    "The appellant was convicted for murder by the Sessions Court.",
    "Section 302 IPC prescribes punishment for murder.",
    "In State v. Mohan this Court held that intent must be proven.",
    "We find the evidence insufficient and allow the appeal.",
]

for sent, label in zip(sentences, predict(sentences)):
    print(f'[{label:<28}]  {sent}')

[Facts                       ]  The appellant was convicted for murder by the Sessions Court.
[Ruling by Lower Court       ]  Section 302 IPC prescribes punishment for murder.
[Ruling by Lower Court       ]  In State v. Mohan this Court held that intent must be proven.
[Facts                       ]  We find the evidence insufficient and allow the appeal.


## Cell 12 — Save trained model files for download

In [24]:
import zipfile

zip_path = '/kaggle/working/trained_model.zip'
files = ['infer/model.tar', 'infer/word2idx.json', 'infer/tag2idx.json']

with zipfile.ZipFile(zip_path, 'w') as zf:
    for f in files:
        full = os.path.join(WORK, f)
        if os.path.exists(full):
            zf.write(full, arcname=os.path.basename(f))
            print('Added:', f)

print(f'\nSaved to {zip_path}')
print('Download from: Kaggle Output tab → trained_model.zip')
print('Next time load with Cell 11 directly — no retraining needed.')

Added: infer/model.tar
Added: infer/word2idx.json
Added: infer/tag2idx.json

Saved to /kaggle/working/trained_model.zip
Download from: Kaggle Output tab → trained_model.zip
Next time load with Cell 11 directly — no retraining needed.
